In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import datetime as dt
from datetime import timedelta

costruzione dataset crude oil da EIA: (frequenze: settimanali, o media giornaliera, mensili)
- produzione 
- stock/reserve di petrolio (inventory)
- OPEC announcement of production estimates
- delta between effective production and OPEC announcement
- demand estimates

U.S. Crude Oil Inventories (total)	PET.WCESTUS1.W	Settimanale
U.S. Crude Oil Production	        PET.WCRPROUS2.W	Settimanale
Cushing OK Inventories	            PET.WGTSTUS1.W	Settimanale
Net U.S. Imports Crude	            PET.WCRIMUS2.W	Settimanale

In [18]:
import pandas as pd
import openpyxl

In [19]:
Stocks = pd.read_excel("C:\\Users\\miche\\OneDrive\\Documenti\\PROJECTS\\crude oil forecasting\\supply factors\\weekly_supply_estimates.xlsx", sheet_name='Data 6')

In [20]:
def clean_date(data, name_column):
    data[name_column] = data[name_column].astype(str)
    data[name_column] = data[name_column].apply(lambda x: x[0:10])

    return data

In [22]:
Stocks = clean_date(Stocks, "Back to Contents")
Reserve = Stocks.iloc[2:,:8].drop(columns = Stocks.iloc[:,1:3])
Reserve = Reserve.drop(columns = Reserve.columns[3:5])
col = ["Date", "Stocks of CL (thousand barrels)", "Stocks of CL excl SPR (thousand barrels)" , "SPR"]

Reserve.columns = col

Reserve = Reserve.set_index("Date")
Reserve


,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR
Date,,,
1982-08-20,609219,338764,270455
1982-08-27,608741,336138,272603
1982-09-24,612419,335586,276833
1982-10-01,612419,334786,277633
1982-10-08,613985,335260,278725
...,...,...,...
2025-06-06,834474,432415,402059
2025-06-13,823231,420942,402289
2025-06-20,817632,415106,402526


In [23]:
prod_oil_w = pd.read_excel("C:\\Users\\miche\\OneDrive\\Documenti\\PROJECTS\\crude oil forecasting\\supply factors\\weekly_supply_estimates.xlsx", sheet_name='Data 1')

In [25]:
prod_oil_w

,Back to Contents,Data 1: Crude Oil Production
0,Sourcekey,WCRFPUS2
1,Date,Weekly U.S. Field Production of Crude Oil (Th...
2,1983-01-07 00:00:00,8634
3,1983-01-14 00:00:00,8634
4,1983-01-21 00:00:00,8634
...,...,...
2212,2025-06-06 00:00:00,13428
2213,2025-06-13 00:00:00,13431
2214,2025-06-20 00:00:00,13435
2215,2025-06-27 00:00:00,13433


In [28]:
#prod_oil_w = clean_date(prod_oil_w, "Back to Contents")

col_p = ["Date", "Crude Oil Production"]
prod_oil_w.columns = col_p


In [29]:
prod_oil_w = prod_oil_w.iloc[2:,:]
prod_oil_w = prod_oil_w.set_index("Date")
prod_oil_w["Crude Oil Production"] = prod_oil_w["Crude Oil Production"]*7

prod_oil_w

,Crude Oil Production
Date,
1983-01-07,60438
1983-01-14,60438
1983-01-21,60438
1983-01-28,60438
1983-02-04,60620
...,...
2025-06-06,93996
2025-06-13,94017
2025-06-20,94045


In [30]:
import xlrd

In [31]:
export = pd.read_excel("C:\\Users\\miche\\OneDrive\\Documenti\\PROJECTS\\crude oil forecasting\\demand factors\\import_export.xls", sheet_name = "Data 2")

In [32]:
export = clean_date(export, "Back to Contents")
col_e = ["Date", "Crude Oil Export"]

In [33]:
export = export.iloc[2:, 0:3]
export.drop(columns = export.columns[1], inplace= True)
export.columns = col_e

export.set_index("Date", inplace= True)
export

,Crude Oil Export
Date,
1991-02-08,138
1991-02-15,138
1991-02-22,242
1991-03-01,242
1991-03-08,167
...,...
2025-10-03,3590
2025-10-10,4466
2025-10-17,4203


In [34]:
dataset = pd.merge(prod_oil_w, Reserve, on = "Date")
dataset = pd.merge(dataset, export, on = "Date")
dataset.index = pd.to_datetime(dataset.index, format="%Y-%m-%d")
dataset

,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-08,52241,896124,311359,584765,138
1991-02-15,51989,900784,317278,583506,138
1991-02-22,51905,899428,316183,583245,242
1991-03-01,51828,897522,315962,581560,242
1991-03-08,51758,892038,314290,577748,167
...,...,...,...,...,...
2025-06-06,93996,834474,432415,402059,3286
2025-06-13,94017,823231,420942,402289,4361
2025-06-20,94045,817632,415106,402526,4270


In [35]:
dataset.to_csv("weekly_crude_oil_supply.csv")

DAILY

In [36]:
daily_dataset = dataset.resample("D").ffill()
daily_dataset


,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-08,52241,896124,311359,584765,138
1991-02-09,52241,896124,311359,584765,138
1991-02-10,52241,896124,311359,584765,138
1991-02-11,52241,896124,311359,584765,138
1991-02-12,52241,896124,311359,584765,138
...,...,...,...,...,...
2025-06-30,94031,821716,418951,402765,2305
2025-07-01,94031,821716,418951,402765,2305
2025-07-02,94031,821716,418951,402765,2305


In [37]:
for col in range(len(daily_dataset.columns)):
    for row in range(len(daily_dataset)):
        daily_dataset.iloc[row, col] = daily_dataset.iloc[row, col] / 4
daily_dataset

,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-08,13060.25,224031.0,77839.75,146191.25,34.5
1991-02-09,13060.25,224031.0,77839.75,146191.25,34.5
1991-02-10,13060.25,224031.0,77839.75,146191.25,34.5
1991-02-11,13060.25,224031.0,77839.75,146191.25,34.5
1991-02-12,13060.25,224031.0,77839.75,146191.25,34.5
...,...,...,...,...,...
2025-06-30,23507.75,205429.0,104737.75,100691.25,576.25
2025-07-01,23507.75,205429.0,104737.75,100691.25,576.25
2025-07-02,23507.75,205429.0,104737.75,100691.25,576.25


In [38]:
daily_dataset.to_csv("daily_oil_data.csv")

MONTHLY

In [40]:
dataset

,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-08,52241,896124,311359,584765,138
1991-02-15,51989,900784,317278,583506,138
1991-02-22,51905,899428,316183,583245,242
1991-03-01,51828,897522,315962,581560,242
1991-03-08,51758,892038,314290,577748,167
...,...,...,...,...,...
2025-06-06,93996,834474,432415,402059,3286
2025-06-13,94017,823231,420942,402289,4361
2025-06-20,94045,817632,415106,402526,4270


In [41]:
monthly_dataset_oil = dataset.resample("ME").mean()
monthly_dataset_oil

,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-28,52045.0,898778.666667,314940.0,583838.666667,172.666667
1991-03-31,51769.2,895377.4,319746.0,575631.4,182.0
1991-04-30,51460.5,892962.5,324480.5,568482.0,122.0
1991-05-31,51151.8,897177.2,328694.4,568482.8,127.0
1991-06-30,51406.25,904637.75,336155.0,568482.75,156.0
...,...,...,...,...,...
2025-03-31,95028.5,832411.0,436402.5,396008.5,4106.0
2025-04-30,94228.75,839613.75,442179.25,397434.5,4003.5
2025-05-31,93737.0,840437.2,439957.2,400480.0,3818.0


In [42]:
for i in range(len(monthly_dataset_oil)):
    monthly_dataset_oil.iloc[i,0] = monthly_dataset_oil.iloc[i,0] * 4   

monthly_dataset_oil

,Crude Oil Production,Stocks of CL (thousand barrels),Stocks of CL excl SPR (thousand barrels),SPR,Crude Oil Export
Date,,,,,
1991-02-28,208180.0,898778.666667,314940.0,583838.666667,172.666667
1991-03-31,207076.8,895377.4,319746.0,575631.4,182.0
1991-04-30,205842.0,892962.5,324480.5,568482.0,122.0
1991-05-31,204607.2,897177.2,328694.4,568482.8,127.0
1991-06-30,205625.0,904637.75,336155.0,568482.75,156.0
...,...,...,...,...,...
2025-03-31,380114.0,832411.0,436402.5,396008.5,4106.0
2025-04-30,376915.0,839613.75,442179.25,397434.5,4003.5
2025-05-31,374948.0,840437.2,439957.2,400480.0,3818.0


In [43]:
monthly_dataset_oil.to_csv("monthly_supply_factors.csv")